# Attach a Mouse Record & Load Dataset — Cloud Workstation

This notebook shows the recommended workflow for a **CodeOcean cloud workstation** session:

1. Load a `MouseRecord` from the dataset catalog.
2. Call `attach_mouse_record_to_workstation()` to attach all round and derived
   assets to the current workstation computation.
3. Build an `HCRDataset` from the now-mounted data.

In [6]:
from pathlib import Path

from aind_hcr_data_loader.codeocean_utils import (
    MouseRecord,
    attach_mouse_record,
    attach_mouse_record_to_workstation,
    print_attach_results,
)
from aind_hcr_data_loader.hcr_dataset import create_hcr_dataset_from_schema

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1 — Configuration

Set the mouse ID and paths.  
`data_dir` is where CodeOcean mounts attached assets (`/root/capsule/data` by convention).

In [7]:
MOUSE_ID = "783884-01"
CATALOG_PATH = Path(f"/src/ophys-mfish-dataset-catalog/mice/{MOUSE_ID}.json")
DATA_DIR = Path("/root/capsule/data")

## 2 — Load the catalog record & attach assets

`MouseRecord.from_json_file()` parses the catalog JSON into a typed dataclass
with `.rounds` and `.derived_assets` dicts.

`attach_mouse_record_to_workstation()` then:
- searches CodeOcean for each asset by name,
- calls `client.computations.attach_data_assets()` with the current
  `CO_COMPUTATION_ID`, and
- returns one `AttachResult` per asset so you can inspect successes/failures.

> **Tip:** pass `dry_run=True` first to verify all assets resolve without
> actually attaching anything.

In [10]:
# Load the catalog record
record = MouseRecord.from_json_file(CATALOG_PATH)

print(f"Mouse:   {record.mouse_id}")
print(f"Rounds:  {list(record.rounds.keys())}")
print(f"Derived: {list(record.derived_assets.keys())}")

Mouse:   783884-01
Rounds:  ['R1', 'R2']
Derived: []


In [11]:
# Optional: dry-run first to confirm all assets are resolvable before attaching
dry_results = attach_mouse_record_to_workstation(record, dry_run=True)
print_attach_results(dry_results, dry_run=True)

  [rounds.R1]  HCR_783884-01_2025-06-27_13-00-00_processed_2025-07-07_23-30-55  →  ✓ found (dry run)
  [rounds.R2]  HCR_783884-01_2025-07-03_13-00-00_processed_2025-07-08_05-34-44  →  ✓ found (dry run)


In [ ]:
# Attach all round + derived assets to this workstation computation.
results = attach_mouse_record_to_workstation(record)
print_attach_results(results)

  [rounds.R1]  HCR_783884-01_2025-06-27_13-00-00_processed_2025-07-07_23-30-55  →  ✓ attached
  [rounds.R2]  HCR_783884-01_2025-07-03_13-00-00_processed_2025-07-08_05-34-44  →  ✓ attached


## 3 — Build the HCRDataset

Now that the assets are mounted under `DATA_DIR`, pass the same catalog record
directly to `create_hcr_dataset_from_schema()`.  It discovers each round
directory automatically from the asset names in the record.

In [13]:
dataset = create_hcr_dataset_from_schema(CATALOG_PATH, DATA_DIR)
dataset.summary()

mouse_id: 783884-01
Could not load metadata for mouse 783884-01
HCR Dataset Summary
Mouse ID: 783884-01
Rounds: R1, R2

Channels by round:
  R1: 405, 488, 561, 638
  R2: 561, 638, 405, 488

Cell-typing: ✗  (not attached)

Segmentation files by round:
  R1: resolutions 0, 2, centroids: ✓
  R2: resolutions 0, 2, centroids: ✗

Spot detection files by round:
  R1: channels 638, 488, 561
    638: spots ✓, stats files: 3
    488: spots ✓, stats files: 3
    561: spots ✓, stats files: 3
  R2: channels 638, 488, 561
    638: spots ✓, stats files: 3
    488: spots ✓, stats files: 3
    561: spots ✓, stats files: 3

Tile alignment files by round:
  R1:
    raw_single_xml: stitching_single_channel_updated_remote.xml
    raw_single_tile_subset_xml: stitching_single_channel_updated_tile_subset_remote.xml
    raw_spot_xml: stitching_spot_channels_updated_remote.xml
    raw_spot_tile_subset_xml: stitching_spot_channels_updated_tile_subset_remote.xml
    pc_xml: bigstitcher.xml
    ip_xml: bigstitcher

## 4 — (Optional) Use the auto-dispatcher

If the same script needs to run in both a workstation *and* a regular capsule,
use `attach_mouse_record()` instead.  It inspects env vars at runtime and
routes to the correct attach function automatically:

| Env var present | Dispatches to |
|---|---|
| `CO_COMPUTATION_ID` | `attach_mouse_record_to_workstation` |
| `CO_CAPSULE_ID` | `attach_mouse_record_to_capsule` |
| neither (explicit `pipeline_id`) | `attach_mouse_record_to_pipeline` |

In [14]:
# Works in a workstation, capsule, or pipeline — no code changes needed.
results = attach_mouse_record(record)
print_attach_results(results)

  [rounds.R1]  HCR_783884-01_2025-06-27_13-00-00_processed_2025-07-07_23-30-55  →  ✓ attached
  [rounds.R2]  HCR_783884-01_2025-07-03_13-00-00_processed_2025-07-08_05-34-44  →  ✓ attached


---
## Quick-start — copy this into your own notebook

```python
from pathlib import Path
from aind_hcr_data_loader.codeocean_utils import (
    MouseRecord,
    attach_mouse_record_to_workstation,
    print_attach_results,
)
from aind_hcr_data_loader.hcr_dataset import create_hcr_dataset_from_schema

# ── configuration ────────────────────────────────────────────────────────────
MOUSE_ID    = "783884-01"
CATALOG_PATH = Path(f"/src/ophys-mfish-dataset-catalog/mice/{MOUSE_ID}.json")
DATA_DIR    = Path("/root/capsule/data")

# ── attach & load ────────────────────────────────────────────────────────────
record  = MouseRecord.from_json_file(CATALOG_PATH)
results = attach_mouse_record_to_workstation(record)
print_attach_results(results)

dataset = create_hcr_dataset_from_schema(CATALOG_PATH, DATA_DIR)
dataset.summary()
```